# 02 · Silver — clean, conform, validate

**Goal:** turn the raw Bronze tables into typed, de-duplicated, validated Delta tables that are safe to
model on. Everything that fails a hard rule is *quarantined* (never silently dropped), and every check
is logged to `silver_dq_results`.

| Problem in the raw data | Treatment |
|---|---|
| Exact duplicate rows | `dropDuplicates()` |
| Stale versions of the same opportunity | keep latest `last_modified_ts` (window `row_number`) |
| `closed won`, `CLOSED LOST`, ` Proposal `, `Closed  Lost` | `clean_label()` → trim, collapse spaces, Title Case |
| `usd` | `upper(trim())` |
| Missing amount | kept and flagged `amount_missing` |
| Negative amount, unknown account / seller / stage | quarantine table with `_reject_reason` |
| `dd/MM/yyyy` mixed with ISO dates | multi-format parser built on `try_to_timestamp` |
| Activities dated before the deal existed | quarantine |

The opportunities table is written with the **Delta MERGE (upsert) pattern**, which is how an
incremental daily load would work.

## 0. Configuration

The same notebook runs unchanged on **Databricks** (Free Edition or any workspace), **Microsoft Fabric**
(attached to a Lakehouse) and a **local PySpark + Delta Lake** session. The platform is auto-detected,
or you can force it with the `LAKEHOUSE_PLATFORM` environment variable.

| Platform | Raw files | Tables |
|---|---|---|
| Databricks | Unity Catalog volume `/Volumes/workspace/sales_lakehouse/raw` | `workspace.sales_lakehouse.<table>` |
| Fabric | Lakehouse `Files/raw` | Lakehouse `Tables` (Delta) |
| Local | `../data/raw` | Spark database `sales_lakehouse` |

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
import os
from datetime import datetime, timezone

def _detect_platform() -> str:
    if os.environ.get("LAKEHOUSE_PLATFORM"):
        return os.environ["LAKEHOUSE_PLATFORM"].lower()
    if "DATABRICKS_RUNTIME_VERSION" in os.environ:
        return "databricks"
    if os.path.isdir("/lakehouse/default"):          # Fabric notebook with a default Lakehouse attached
        return "fabric"
    return "local"

PLATFORM = _detect_platform()            # "databricks" | "fabric" | "local"
CATALOG  = "workspace"                   # Databricks Unity Catalog (Free Edition default catalog)
SCHEMA   = "sales_lakehouse"             # Databricks schema / local Spark database

if PLATFORM == "databricks":
    RAW_PATH    = f"/Volumes/{CATALOG}/{SCHEMA}/raw"
    EXPORT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/export"
    def tbl(name: str) -> str:
        return f"{CATALOG}.{SCHEMA}.{name}"
elif PLATFORM == "fabric":
    RAW_PATH    = "Files/raw"
    EXPORT_PATH = "Files/export"
    def tbl(name: str) -> str:
        return name                       # tables live in the attached Lakehouse
else:
    RAW_PATH    = os.path.abspath("../data/raw")
    EXPORT_PATH = os.path.abspath("../lakehouse/export")
    def tbl(name: str) -> str:
        return f"{SCHEMA}.{name}"

try:
    spark  # noqa: F821 - pre-defined on Databricks and Fabric
except NameError:                          # local run: build a Delta-enabled SparkSession
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    _builder = (SparkSession.builder.appName("sales-pipeline-lakehouse")
                .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
                .config("spark.sql.warehouse.dir", os.path.abspath("../lakehouse/warehouse")))
    spark = configure_spark_with_delta_pip(_builder).getOrCreate()

if PLATFORM == "databricks":
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
    for _vol in ("raw", "export"):
        try:
            spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{_vol}")
        except Exception as _e:              # no privilege → create the volume in Catalog Explorer instead
            print(f"could not create volume '{_vol}': {str(_e).splitlines()[0]}")
elif PLATFORM == "local":
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {SCHEMA}")

RUN_TS = datetime.now(timezone.utc)
print(f"platform={PLATFORM} | raw={RAW_PATH} | export={EXPORT_PATH} | spark={spark.version}")

## 1. Helpers

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

def clean_label(col: str):
    """'  closed  WON ' -> 'Closed Won' : trim, collapse inner whitespace, Title Case."""
    return F.initcap(F.lower(F.regexp_replace(F.trim(F.col(col)), r"\s+", " ")))

def parse_date(col: str):
    """Parse a date that may arrive as ISO (yyyy-MM-dd) or European (dd/MM/yyyy); null if neither."""
    c = F.trim(F.col(col))
    return F.coalesce(
        F.try_to_timestamp(c, F.lit("yyyy-MM-dd")),
        F.try_to_timestamp(c, F.lit("dd/MM/yyyy")),
        F.try_to_timestamp(c, F.lit("yyyy-MM-dd HH:mm:ss")),
    ).cast("date")

def parse_ts(col: str):
    return F.try_to_timestamp(F.trim(F.col(col)), F.lit("yyyy-MM-dd HH:mm:ss"))

dq_results = []                                   # (check_name, table_name, failed_rows, total_rows, status)

def dq_check(name: str, table: str, failed: int, total: int, warn_threshold_pct: float = 0.0):
    """Record a data-quality check. PASS = no failures, WARN = within tolerance, FAIL = above tolerance."""
    pct = (failed / total * 100) if total else 0.0
    status = "PASS" if failed == 0 else ("WARN" if pct <= warn_threshold_pct else "FAIL")
    dq_results.append((name, table, int(failed), int(total), status))
    print(f"[{status:4}] {name:<52} {failed:>6,} / {total:,}  ({pct:.2f}%)")

def upsert_delta(df, table_name: str, key_cols):
    """First run: create the table. Later runs: MERGE (upsert) on the business key — the incremental pattern."""
    full_name = tbl(table_name)
    if spark.catalog.tableExists(full_name):
        cond = " AND ".join(f"t.{k} = s.{k}" for k in key_cols)
        (DeltaTable.forName(spark, full_name).alias("t")
             .merge(df.alias("s"), cond)
             .whenMatchedUpdateAll()
             .whenNotMatchedInsertAll()
             .execute())
        action = "merged into"
    else:
        df.write.format("delta").mode("overwrite").saveAsTable(full_name)
        action = "created"
    print(f"{action} {full_name}: {spark.table(full_name).count():,} rows")

## 2. Sellers and accounts (dimension sources)

In [ ]:
silver_sellers = (spark.table(tbl("bronze_sellers"))
    .select(
        F.trim("seller_id").alias("seller_id"),
        F.trim("seller_name").alias("seller_name"),
        F.trim("region").alias("region"),
        F.trim("segment").alias("segment"),
        F.trim("team").alias("team"),
        F.trim("manager_name").alias("manager_name"),
        parse_date("hire_date").alias("hire_date"),
        F.col("quota_annual").cast("decimal(12,2)").alias("quota_annual"),
        F.col("is_active").cast("boolean").alias("is_active"),
    )
    .dropDuplicates(["seller_id"]))

silver_accounts = (spark.table(tbl("bronze_accounts"))
    .select(
        F.trim("account_id").alias("account_id"),
        F.trim("account_name").alias("account_name"),
        F.trim("industry").alias("industry"),
        F.trim("country").alias("country"),
        F.trim("region").alias("region"),
        F.trim("segment").alias("segment"),
        F.trim("employee_band").alias("employee_band"),
        parse_date("created_date").alias("created_date"),
        F.trim("owner_seller_id").alias("owner_seller_id"),
    )
    .dropDuplicates(["account_id"]))

dq_check("sellers: quota_annual missing", "silver_sellers", silver_sellers.filter("quota_annual IS NULL").count(), silver_sellers.count())
dq_check("accounts: owner_seller_id unknown", "silver_accounts",
         silver_accounts.join(silver_sellers.select(F.col("seller_id").alias("owner_seller_id")), "owner_seller_id", "left_anti").count(),
         silver_accounts.count())

silver_sellers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("silver_sellers"))
silver_accounts.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("silver_accounts"))
print("silver_sellers:", silver_sellers.count(), "| silver_accounts:", silver_accounts.count())

## 3. Opportunities — de-duplicate, type, normalise

In [ ]:
bronze_opps = spark.table(tbl("bronze_opportunities"))
n_bronze = bronze_opps.count()

# 1) exact duplicate rows (compare business columns only, not the lineage columns)
business_cols = [c for c in bronze_opps.columns if not c.startswith("_")]
deduped = bronze_opps.dropDuplicates(business_cols)
dq_check("opportunities: exact duplicate rows removed", "bronze_opportunities", n_bronze - deduped.count(), n_bronze, warn_threshold_pct=5)

# 2) typing + normalisation
typed = deduped.select(
    F.trim("opportunity_id").alias("opportunity_id"),
    F.trim("opportunity_name").alias("opportunity_name"),
    F.trim("account_id").alias("account_id"),
    F.trim("seller_id").alias("seller_id"),
    F.trim("product").alias("product"),
    clean_label("lead_source").alias("lead_source"),
    clean_label("stage").alias("stage"),                              # 'closed  WON ' -> 'Closed Won'
    F.col("probability").cast("decimal(4,2)").alias("probability"),
    F.col("amount").cast("decimal(12,2)").alias("amount"),
    F.upper(F.trim("currency")).alias("currency"),                    # 'usd' -> 'USD'
    parse_date("created_date").alias("created_date"),
    parse_date("expected_close_date").alias("expected_close_date"),   # ISO or dd/MM/yyyy
    parse_date("actual_close_date").alias("actual_close_date"),
    parse_ts("last_modified_ts").alias("last_modified_ts"),
    F.col("_ingest_ts"),
    F.col("_batch_id"),
)

# 3) keep only the latest version of each opportunity
w_latest = Window.partitionBy("opportunity_id").orderBy(F.col("last_modified_ts").desc_nulls_last())
versioned = typed.withColumn("_rn", F.row_number().over(w_latest))
dq_check("opportunities: stale versions superseded", "bronze_opportunities", versioned.filter("_rn > 1").count(), versioned.count(), warn_threshold_pct=5)
latest = versioned.filter("_rn = 1").drop("_rn")

print("Stage labels after normalisation:")
latest.groupBy("stage").count().orderBy("stage").show(truncate=False)

## 4. Opportunities — validate and quarantine

Hard-rule failures go to `silver_opportunities_quarantine` with a `_reject_reason`; soft issues
(missing amount) stay in Silver but are flagged.

In [ ]:
VALID_STAGES = ["Prospecting", "Qualification", "Proposal", "Negotiation", "Closed Won", "Closed Lost"]

checked = (latest
    .join(silver_accounts.select("account_id").withColumn("_acct_ok", F.lit(True)), "account_id", "left")
    .join(silver_sellers.select("seller_id").withColumn("_seller_ok", F.lit(True)), "seller_id", "left")
    .withColumn("amount_missing", F.col("amount").isNull())
    .withColumn("_reject_reason", F.concat_ws("; ",
        F.when(F.col("_acct_ok").isNull(), F.lit("unknown account_id")),
        F.when(F.col("_seller_ok").isNull(), F.lit("unknown seller_id")),
        F.when(F.col("amount") < 0, F.lit("negative amount")),
        F.when(~F.col("stage").isin(VALID_STAGES), F.lit("unknown stage")),
        F.when(F.col("created_date").isNull(), F.lit("unparseable created_date")),
    ))
    .drop("_acct_ok", "_seller_ok"))

n_latest = checked.count()
for reason in ["unknown account_id", "unknown seller_id", "negative amount", "unknown stage", "unparseable created_date"]:
    dq_check(f"opportunities: {reason}", "silver_opportunities",
             checked.filter(F.col("_reject_reason").contains(reason)).count(), n_latest, warn_threshold_pct=1)

quarantine = checked.filter(F.col("_reject_reason") != "").withColumn("_quarantined_ts", F.lit(RUN_TS))
clean = checked.filter(F.col("_reject_reason") == "").drop("_reject_reason")

dq_check("opportunities: amount missing (kept, flagged)", "silver_opportunities", clean.filter("amount_missing").count(), n_latest, warn_threshold_pct=3)
dq_check("opportunities: expected_close_date unparseable", "silver_opportunities", clean.filter("expected_close_date IS NULL").count(), n_latest)
dq_check("opportunities: closed deal without actual_close_date", "silver_opportunities",
         clean.filter(F.col("stage").isin("Closed Won", "Closed Lost") & F.col("actual_close_date").isNull()).count(), n_latest)

silver_opportunities = (clean
    .withColumn("is_closed", F.col("stage").isin("Closed Won", "Closed Lost"))
    .withColumn("is_won", F.col("stage") == "Closed Won")
    .withColumn("sales_cycle_days", F.when(F.col("stage").isin("Closed Won", "Closed Lost"),
                                           F.datediff("actual_close_date", "created_date")))
    .withColumn("weighted_amount", (F.col("amount") * F.col("probability")).cast("decimal(12,2)"))
    .withColumn("_silver_ts", F.lit(RUN_TS)))

upsert_delta(silver_opportunities, "silver_opportunities", key_cols=["opportunity_id"])
quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("silver_opportunities_quarantine"))
print("quarantined:", quarantine.count())
quarantine.select("opportunity_id", "account_id", "stage", "amount", "_reject_reason").show(10, truncate=False)

## 5. Stage history and activities

In [ ]:
# --- stage history --------------------------------------------------------------
hist_raw = spark.table(tbl("bronze_opportunity_stage_history"))
hist = (hist_raw.select(
            F.trim("history_id").alias("history_id"),
            F.trim("opportunity_id").alias("opportunity_id"),
            clean_label("from_stage").alias("from_stage"),
            clean_label("to_stage").alias("to_stage"),
            parse_ts("changed_at").alias("changed_at"),
            F.trim("changed_by_seller_id").alias("changed_by_seller_id"))
        .dropDuplicates(["history_id"]))

opp_keys = silver_opportunities.select("opportunity_id", "created_date")
hist_orphans = hist.join(opp_keys, "opportunity_id", "left_anti").count()
dq_check("stage_history: rows for quarantined/unknown opportunities (dropped)", "silver_opportunity_stage_history",
         hist_orphans, hist.count(), warn_threshold_pct=2)
silver_history = hist.join(opp_keys.select("opportunity_id"), "opportunity_id", "inner").withColumn("_silver_ts", F.lit(RUN_TS))
silver_history.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("silver_opportunity_stage_history"))
print("silver_opportunity_stage_history:", silver_history.count())

# --- activities -----------------------------------------------------------------
acts = (spark.table(tbl("bronze_activities")).select(
            F.trim("activity_id").alias("activity_id"),
            F.trim("opportunity_id").alias("opportunity_id"),
            F.trim("seller_id").alias("seller_id"),
            clean_label("activity_type").alias("activity_type"),
            parse_ts("activity_ts").alias("activity_ts"),
            F.col("duration_minutes").cast("int").alias("duration_minutes"))
        .dropDuplicates(["activity_id"]))

acts_checked = (acts.join(opp_keys, "opportunity_id", "left")
    .withColumn("_reject_reason", F.concat_ws("; ",
        F.when(F.col("created_date").isNull(), F.lit("unknown opportunity_id")),
        F.when(F.to_date("activity_ts") < F.col("created_date"), F.lit("activity before opportunity created")),
    )))
n_acts = acts_checked.count()
for reason in ["unknown opportunity_id", "activity before opportunity created"]:
    dq_check(f"activities: {reason}", "silver_activities",
             acts_checked.filter(F.col("_reject_reason").contains(reason)).count(), n_acts, warn_threshold_pct=2)

acts_quarantine = acts_checked.filter(F.col("_reject_reason") != "").withColumn("_quarantined_ts", F.lit(RUN_TS))
silver_activities = (acts_checked.filter(F.col("_reject_reason") == "")
                        .drop("_reject_reason", "created_date")
                        .withColumn("_silver_ts", F.lit(RUN_TS)))
silver_activities.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("silver_activities"))
acts_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("silver_activities_quarantine"))
print("silver_activities:", silver_activities.count(), "| quarantined:", acts_quarantine.count())

## 6. Persist the data-quality results

Appending one row per check per run turns data quality into a time series you can chart in Power BI.

In [ ]:
dq_df = (spark.createDataFrame(dq_results, "check_name string, table_name string, failed_rows long, total_rows long, status string")
    .withColumn("failed_pct", F.round(F.col("failed_rows") / F.col("total_rows") * 100, 2))
    .withColumn("run_ts", F.lit(RUN_TS))
    .withColumn("platform", F.lit(PLATFORM)))

dq_df.write.format("delta").mode("append").saveAsTable(tbl("silver_dq_results"))
dq_df.select("status", "check_name", "failed_rows", "total_rows", "failed_pct").orderBy("status", "check_name").show(50, truncate=False)

if dq_df.filter("status = 'FAIL'").count() > 0:
    raise RuntimeError("Data-quality FAIL — see silver_dq_results before running Gold")

## 7. Delta housekeeping and time travel

In [ ]:
for t in ["silver_opportunities", "silver_opportunity_stage_history", "silver_activities"]:
    try:
        spark.sql(f"OPTIMIZE {tbl(t)}")               # compacts small files (Databricks / Fabric / Delta 2.0+)
        print("optimized", t)
    except Exception as e:                             # older local Delta versions
        print("skipped OPTIMIZE for", t, "-", str(e).splitlines()[0])

# time travel: compare the current row count with the first version of the table
hist_tbl = spark.sql(f"DESCRIBE HISTORY {tbl('silver_opportunities')}").select("version", "timestamp", "operation")
hist_tbl.show(truncate=False)
v0 = spark.read.format("delta").option("versionAsOf", 0).table(tbl("silver_opportunities")).count()
print(f"silver_opportunities @version 0: {v0:,} rows | current: {spark.table(tbl('silver_opportunities')).count():,} rows")

✅ **Silver complete.** Continue with `03_gold_star_schema.ipynb`.